In [1]:
!pip install -q ultralytics faster-coco-eval
# !mkdir -p /kaggle/working/data
# !cp -r /kaggle/input/ripvis-cs431/datasets/train /kaggle/working/data/train

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 31.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import json
import yaml
import os
from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils
import torch
import random

FOLD_CSV = '/kaggle/input/ripvis-cs431/fold_management.csv'
DATASET_PATH = '/kaggle/working/data/train'
TEST_PATH = '/kaggle/working/data/test'
IOU_THRESHOLDS = np.arange(0.40, 0.96, 0.05)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [4]:
def compute_composite_score(gt_json_path, pred_json_path):
    if not os.path.exists(pred_json_path): return {"Score": 0.0}
    
    gt_coco = COCO(gt_json_path)
    with open(pred_json_path) as f: preds = json.load(f)
    pred_coco = gt_coco.loadRes(preds)
    
    # Evaluate SEG
    counts = [[0, 0, 0] for _ in IOU_THRESHOLDS]
    for img_id in gt_coco.getImgIds():
        gt_anns = [a for a in gt_coco.loadAnns(gt_coco.getAnnIds(imgIds=img_id)) if "segmentation" in a]
        pr_anns = sorted([a for a in pred_coco.loadAnns(pred_coco.getAnnIds(imgIds=img_id)) if "segmentation" in a], 
                         key=lambda x: x.get("score", 0), reverse=True)
        
        if not gt_anns and not pr_anns: continue
        
        # IoU Matching logic (RLE)
        gt_rle = [gt_coco.annToRLE(a) for a in gt_anns]
        pr_rle = [gt_coco.annToRLE(a) for a in pr_anns]
        
        if not pr_anns:
            for idx in range(len(IOU_THRESHOLDS)): counts[idx][2] += len(gt_anns)
            continue
        if not gt_anns:
            for idx in range(len(IOU_THRESHOLDS)): counts[idx][1] += len(pr_anns)
            continue

        ious = maskUtils.iou(pr_rle, gt_rle, np.zeros(len(gt_rle)))
        for idx, thr in enumerate(IOU_THRESHOLDS):
            matched_gt = np.zeros(len(gt_anns), dtype=bool)
            tp = fp = 0
            for i in range(len(pr_anns)):
                eligible = (ious[i] >= thr) & (~matched_gt)
                if eligible.any():
                    tp += 1
                    matched_gt[np.argmax(np.where(eligible, ious[i], -1.0))] = True
                else: fp += 1
            counts[idx][0] += tp; counts[idx][1] += fp; counts[idx][2] += int((~matched_gt).sum())

    # Calculate Metrics
    f1_list, f2_list = [], []
    for tp, fp, fn in counts:
        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_list.append(2*p*r/(p+r) if (p+r)>0 else 0)
        f2_list.append(5*p*r/(4*p+r) if (4*p+r)>0 else 0)

    # Lấy F1 và F2 tại ngưỡng IoU = 0.50 (index 2 trong mảng 0.40, 0.45, 0.50...)
    idx_50 = 2
    f1_50 = f1_list[idx_50]
    f2_50 = f2_list[idx_50]
    
    # Tính mean cho toàn bộ ngưỡng (0.40 đến 0.95)
    f1_4095 = float(np.mean(f1_list))
    f2_4095 = float(np.mean(f2_list))

    # Composite: 0.25*(F1@50 + F2@50 + mean_F1 + mean_F2)
    score = 0.25 * f1_50 + 0.25 * f2_50 + 0.25 * f1_4095 + 0.25 * f2_4095

    return {
        "F1[50]": float(f1_50),
        "F2[50]": float(f2_50),
        "F1[40:95]": float(f1_4095),
        "F2[40:95]": float(f2_4095),
        "Score": float(score)
    }

In [5]:
def run_training_for_fold(fold_num):
    df = pd.read_csv(FOLD_CSV)
    train_df = df[df['fold'] != fold_num]
    val_df = df[df['fold'] == fold_num]

    # Tạo file list cho YOLO
    train_list = f'train_f{fold_num}.txt'
    val_list = f'val_f{fold_num}.txt'
    
    with open(train_list, 'w') as f:
        f.write('\n'.join([os.path.join(DATASET_PATH, 'images', name) for name in train_df['file_name']]))
    with open(val_list, 'w') as f:
        f.write('\n'.join([os.path.join(DATASET_PATH, 'images', name) for name in val_df['file_name']]))

    # Tạo GT JSON cho Fold này để đánh giá
    with open(os.path.join(DATASET_PATH, 'train_with_additional_data.json'), 'r') as f:
        full_coco = json.load(f)
    
    val_fns = set(val_df['file_name'])
    fold_gt = {
        "images": [i for i in full_coco['images'] if i['file_name'] in val_fns],
        "annotations": [],
        "categories": full_coco['categories']
    }
    v_ids = set(i['id'] for i in fold_gt['images'])
    fold_gt['annotations'] = [a for a in full_coco['annotations'] if a['image_id'] in v_ids]
    
    gt_json_fold = f'gt_fold_{fold_num}.json'
    with open(gt_json_fold, 'w') as f:
        json.dump(fold_gt, f)

    # Cấu hình YAML
    yaml_p = f'rip_f{fold_num}.yaml'
    with open(yaml_p, 'w') as f:
        yaml.dump({'path': '/kaggle/working', 'train': train_list, 'val': val_list, 'nc': 1, 'names': ['rip']}, f)

    # Huấn luyện
    model = YOLO('yolov8n-seg.pt')
    !yolo task=segment mode=train \
        model=yolov8n-seg.pt \
        data={yaml_p} \
        epochs=150 imgsz=640 \
        batch=200 \
        workers=2 \
        device=0,1 \
        seed=42 \
        cache=False \
        project=Rip_Project name=fold_{fold_num}
    model = YOLO(f'/kaggle/working/runs/segment/Rip_Project/fold_{fold_num}/weights/best.pt')
    model.val(data=yaml_p, split='val', save_json=True, project='Eval', name=f'fold_{fold_num}', conf=0.25)

    pred_json_path = f'/kaggle/working/runs/segment/Eval/fold_{fold_num}/predictions.json'
    
    # Kiểm tra xem YOLO có sinh ra file không
    if not os.path.exists(pred_json_path):
        print(f"Không tìm thấy {pred_json_path}. Trả về Score 0.0")
        score_results = {"Score": 0.0, "F1[50]": 0.0, "F2[50]": 0.0, "F1[40:95]": 0.0, "F2[40:95]": 0.0}
    else:
        # 1. Tạo từ điển map Tên file -> ID số từ biến fold_gt đã tạo ở trên
        filename_to_id = {os.path.splitext(img['file_name'])[0]: img['id'] for img in fold_gt['images']}
        
        # 2. Mở file JSON dự đoán
        with open(pred_json_path, 'r') as f:
            preds = json.load(f)
            
        # 3. Đổi ID chữ thành ID số
        valid_preds = []
        for p in preds:
            stem = str(p['image_id']) # ví dụ: "RipVIS-141_00060"
            if stem in filename_to_id:
                p['image_id'] = filename_to_id[stem] # Gán lại ID số nguyên
                valid_preds.append(p)
                
        # 4. Ghi đè lại file
        with open(pred_json_path, 'w') as f:
            json.dump(valid_preds, f)
            
        # 5. Đưa file ĐÃ SỬA vào hàm tính điểm
        score_results = compute_composite_score(gt_json_fold, pred_json_path)
        
    score_results['fold'] = fold_num

    json_result_path = f'result_fold_{fold_num}.json'
    with open(json_result_path, 'w') as f:
        json.dump(score_results, f, indent=4)
        
    return score_results

In [6]:
# current_fold = 6
# result = run_training_for_fold(current_fold)
# print(f"Kết quả Fold {current_fold}: {result}")

In [7]:
import yaml

def create_test_yaml(test_images_path, yaml_path):
    # Tạo file list .txt cho tập test
    test_list_path = 'test_images_list.txt'
    image_files = [os.path.join(test_images_path, f) for f in os.listdir(test_images_path) 
                   if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    with open(test_list_path, 'w') as f:
        f.write('\n'.join(image_files))
    
    # Tạo file YAML
    test_data = {
        'path': '/kaggle/working',
        'train': test_list_path, # YOLO yêu cầu có train/val, ta để tạm cùng test_list
        'val': test_list_path,
        'test': test_list_path,
        'nc': 1,
        'names': ['rip']
    }
    with open(yaml_path, 'w') as f:
        yaml.dump(test_data, f)
    
    return test_list_path

In [8]:
def run_test_and_evaluate(model_path, test_images_path, gt_json_path, save_result_path):
    # 1. Tạo YAML cho tập Test
    test_yaml = 'test_config.yaml'
    create_test_yaml(test_images_path, test_yaml)
    
    # 2. Load Model và chạy Validation trên tập Test
    model = YOLO(model_path)
    # Dùng model.val để YOLO tự xuất file predictions.json chuẩn
    model.val(data=test_yaml, split='val', save_json=True, project='Test_Eval', name='final_test', conf=0.25)
    
    # Đường dẫn file mặc định YOLO tạo ra
    yolo_pred_path = '/kaggle/working/runs/segment/Test_Eval/final_test/predictions.json'
    
    if not os.path.exists(yolo_pred_path):
        print("Lỗi: YOLO không xuất được file predictions.json")
        return None

    # 3. Thực hiện Mapping ID (Bê nguyên logic từ code Train của bạn)
    # Load GT tập test để lấy mapping
    gt_coco = COCO(gt_json_path)
    # Lưu ý: Đoạn này phải khớp với cách bạn làm trong Train (splitext hay không)
    filename_to_id = {os.path.splitext(img['file_name'])[0]: img['id'] for img in gt_coco.imgs.values()}
    
    with open(yolo_pred_path, 'r') as f:
        preds = json.load(f)
        
    valid_preds = []
    for p in preds:
        stem = str(p['image_id']) # YOLO lấy tên file làm ID
        if stem in filename_to_id:
            p['image_id'] = filename_to_id[stem] # Đổi thành ID số nguyên
            valid_preds.append(p)
    
    # 4. Ghi đè lại file đã sửa ID
    with open(save_result_path, 'w') as f:
        json.dump(valid_preds, f)
        
    print(f"--- Đã đồng bộ ID cho {len(valid_preds)} dự đoán tập Test ---")

    # 5. Tính Score bằng hàm của bạn
    return compute_composite_score(gt_json_path, save_result_path)

In [9]:
MODEL_PATH = '/kaggle/input/datasets/nttgaming/best-model-yolov8-fold0/best.pt' 
TEST_IMAGES = '/kaggle/input/ripvis-cs431/datasets/test/images'
GT_JSON = '/kaggle/input/ripvis-cs431/datasets/test/test_gt.json'
SAVE_JSON = 'predictions_test.json'

metrics = run_test_and_evaluate(MODEL_PATH, TEST_IMAGES, GT_JSON, SAVE_JSON)
score_file_path = 'final_test_scores.json'

with open(score_file_path, 'w') as f:
    json.dump(metrics, f, indent=4)
    
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 2.4±0.3 ms, read: 39.5±24.6 MB/s, size: 220.9 KB)
val: Scanning /kaggle/input/ripvis-cs431/datasets/test/labels... 2861 images, 1488 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4349/4349 290.5it/s 15.0s
WARNING ⚠️ val: Cache directory /kaggle/input/ripvis-cs431/datasets/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 272/272 6.0it/s 45.6s
                   all       4349       4556        0.8      0.454      0.416      0.183      0.801      0.454      0.421      0.163
Speed: 0.6ms preprocess, 2.9ms inference, 0.0ms loss, 0.6ms postprocess per image
Saving /kaggle/working/runs/segment/Test_Eval/final_test/predictions.json...
Results saved to /kaggle